In [18]:
import pandas as pd
calendar = pd.read_csv("data/raw/calendar.csv")
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


In [19]:
calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 1969 entries, 0 to 1968
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   date          1969 non-null   str  
 1   wm_yr_wk      1969 non-null   int64
 2   weekday       1969 non-null   str  
 3   wday          1969 non-null   int64
 4   month         1969 non-null   int64
 5   year          1969 non-null   int64
 6   d             1969 non-null   str  
 7   event_name_1  162 non-null    str  
 8   event_type_1  162 non-null    str  
 9   event_name_2  5 non-null      str  
 10  event_type_2  5 non-null      str  
 11  snap_CA       1969 non-null   int64
 12  snap_TX       1969 non-null   int64
 13  snap_WI       1969 non-null   int64
dtypes: int64(7), str(7)
memory usage: 215.5 KB


In [20]:
calendar["date"] = pd.to_datetime(
    calendar["date"],
    format="%Y-%m-%d"
)

calendar["date"].dtype

dtype('<M8[us]')

In [21]:
print("Prima data:", calendar["date"].min())
print("Ultima data:", calendar["date"].max())
print("Date duplicate:", calendar["date"].duplicated().sum())

Prima data: 2011-01-29 00:00:00
Ultima data: 2016-06-19 00:00:00
Date duplicate: 0


In [22]:
print("Codici giorno mancanti:", calendar["d"].isna().sum())
print("Codici giorno duplicati:", calendar["d"].duplicated().sum())

Codici giorno mancanti: 0
Codici giorno duplicati: 0


In [23]:
sales_sample = pd.read_csv(
    "data/raw/sales_train_evaluation.csv",
    nrows=5
)

sales_sample

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


In [24]:
sales_sample.columns[-5:].tolist()

['d_1937', 'd_1938', 'd_1939', 'd_1940', 'd_1941']

In [25]:
calendar.loc[
    calendar["d"] == "d_1941",
    ["d", "date"]
]

,d,date
1940,d_1941,2016-05-22


In [26]:
products = pd.read_csv(
    "data/raw/sales_train_evaluation.csv",
    usecols=["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
)

print("Numero di righe:", len(products))
print("Negozi:", products["store_id"].unique())
print("Categorie:", products["cat_id"].unique())

Numero di righe: 30490
Negozi: <StringArray>
['CA_1', 'CA_2', 'CA_3', 'CA_4', 'TX_1', 'TX_2', 'TX_3', 'WI_1', 'WI_2',
 'WI_3']
Length: 10, dtype: str
Categorie: <StringArray>
['HOBBIES', 'HOUSEHOLD', 'FOODS']
Length: 3, dtype: str


In [27]:
print(
    "Combinazioni prodotto-negozio duplicate:",
    products.duplicated(subset=["item_id", "store_id"]).sum()
)

Combinazioni prodotto-negozio duplicate: 0


In [28]:
products_scope = products.loc[
    (products["store_id"] == "CA_1") &
    (products["cat_id"] == "HOUSEHOLD")
].copy()

print("Prodotti nel perimetro:", len(products_scope))
products_scope.head()

Prodotti nel perimetro: 1047


,id,item_id,dept_id,cat_id,store_id,state_id
565,HOUSEHOLD_1_001_CA_1_evaluation,HOUSEHOLD_1_001,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA
566,HOUSEHOLD_1_002_CA_1_evaluation,HOUSEHOLD_1_002,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA
567,HOUSEHOLD_1_003_CA_1_evaluation,HOUSEHOLD_1_003,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA
568,HOUSEHOLD_1_004_CA_1_evaluation,HOUSEHOLD_1_004,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA
569,HOUSEHOLD_1_005_CA_1_evaluation,HOUSEHOLD_1_005,HOUSEHOLD_1,HOUSEHOLD,CA_1,CA


In [29]:
selected_blocks = []

for block in pd.read_csv(
    "data/raw/sales_train_evaluation.csv",
    chunksize=3000
):
    selected = block.loc[
        block["id"].isin(products_scope["id"])
    ].copy()
    selected_blocks.append(selected)

sales_scope = pd.concat(selected_blocks, ignore_index=True)

print("Dimensioni:", sales_scope.shape)

Dimensioni: (1047, 1947)


In [30]:
day_columns = sales_scope.columns[
    sales_scope.columns.str.startswith("d_")
].tolist()

print("Numero di giorni:", len(day_columns))
print("Primi 3 giorni:", day_columns[:3])
print("Ultimi 3 giorni:", day_columns[-3:])

Numero di giorni: 1941
Primi 3 giorni: ['d_1', 'd_2', 'd_3']
Ultimi 3 giorni: ['d_1939', 'd_1940', 'd_1941']


In [31]:
daily_sales = sales_scope[day_columns]

print("Valori mancanti:", daily_sales.isna().sum().sum())
print("Valori negativi:", (daily_sales < 0).sum().sum())

Valori mancanti: 0
Valori negativi: 0


In [32]:
sales_long = sales_scope.melt(
    id_vars=["item_id", "store_id"],
    value_vars=day_columns,
    var_name="d",
    value_name="units_sold"
)

sales_long.head()

,item_id,store_id,d,units_sold
0,HOUSEHOLD_1_001,CA_1,d_1,3
1,HOUSEHOLD_1_002,CA_1,d_1,1
2,HOUSEHOLD_1_003,CA_1,d_1,0
3,HOUSEHOLD_1_004,CA_1,d_1,4
4,HOUSEHOLD_1_005,CA_1,d_1,0


In [33]:
sales_long = sales_long.merge(
    calendar[["d", "date"]],
    on="d",
    how="left",
    validate="many_to_one"
)

sales_long.head()

,item_id,store_id,d,units_sold,date
0,HOUSEHOLD_1_001,CA_1,d_1,3,2011-01-29
1,HOUSEHOLD_1_002,CA_1,d_1,1,2011-01-29
2,HOUSEHOLD_1_003,CA_1,d_1,0,2011-01-29
3,HOUSEHOLD_1_004,CA_1,d_1,4,2011-01-29
4,HOUSEHOLD_1_005,CA_1,d_1,0,2011-01-29


In [34]:
print("Date mancanti:", sales_long["date"].isna().sum())
print("Numero di righe:", len(sales_long))

Date mancanti: 0
Numero di righe: 2032227


In [35]:
print("Prima data delle vendite:", sales_long["date"].min())
print("Ultima data delle vendite:", sales_long["date"].max())

Prima data delle vendite: 2011-01-29 00:00:00
Ultima data delle vendite: 2016-05-22 00:00:00


In [36]:
test_start = pd.Timestamp("2016-04-25")

sales_dev = sales_long.loc[
    sales_long["date"] < test_start
].copy()

sales_test = sales_long.loc[
    sales_long["date"] >= test_start
].copy()

print("Giorni di sviluppo:", sales_dev["date"].nunique())
print("Giorni di test:", sales_test["date"].nunique())

Giorni di sviluppo: 1913
Giorni di test: 28


In [37]:
sales_2015 = sales_dev.loc[
    (sales_dev["date"] >= "2015-01-01") &
    (sales_dev["date"] < "2016-01-01")
].copy()

product_ranking = (
    sales_2015.groupby("item_id")["units_sold"]
    .sum()
    .sort_values(ascending=False)
)

product_ranking.head(20)

item_id
HOUSEHOLD_1_110    4149
HOUSEHOLD_1_334    3803
HOUSEHOLD_1_418    3534
HOUSEHOLD_1_118    3296
HOUSEHOLD_1_066    3259
HOUSEHOLD_1_517    3087
HOUSEHOLD_1_514    3028
HOUSEHOLD_1_465    2812
HOUSEHOLD_1_096    2689
HOUSEHOLD_1_535    2645
HOUSEHOLD_1_399    2644
HOUSEHOLD_1_447    2578
HOUSEHOLD_1_459    2556
HOUSEHOLD_1_409    2425
HOUSEHOLD_1_303    2257
HOUSEHOLD_1_191    2179
HOUSEHOLD_1_521    2148
HOUSEHOLD_1_351    2101
HOUSEHOLD_1_151    2079
HOUSEHOLD_1_304    2055
Name: units_sold, dtype: int64

In [38]:
selected_items = product_ranking.head(20).index.tolist()

sales_dev_20 = sales_dev.loc[
    sales_dev["item_id"].isin(selected_items)
].copy()

print("Prodotti selezionati:", sales_dev_20["item_id"].nunique())
print("Righe dello storico:", len(sales_dev_20))

Prodotti selezionati: 20
Righe dello storico: 38260


In [39]:
import sqlite3

conn = sqlite3.connect(":memory:")

sales_dev_20.to_sql(
    "sales_daily",
    conn,
    index=False,
    if_exists="replace"
)

38260

In [40]:
query_weekly = """
SELECT
    item_id,
    store_id,
    DATE(date, '-6 days', 'weekday 1') AS week_start,
    SUM(units_sold) AS weekly_units,
    COUNT(DISTINCT DATE(date)) AS days_present
FROM sales_daily
GROUP BY item_id, store_id, week_start
HAVING COUNT(DISTINCT DATE(date)) = 7
ORDER BY item_id, store_id, week_start
"""

sales_weekly = pd.read_sql_query(
    query_weekly,
    conn,
    parse_dates=["week_start"]
)

sales_weekly.head()

,item_id,store_id,week_start,weekly_units,days_present
0,HOUSEHOLD_1_066,CA_1,2011-01-31,0,7
1,HOUSEHOLD_1_066,CA_1,2011-02-07,0,7
2,HOUSEHOLD_1_066,CA_1,2011-02-14,0,7
3,HOUSEHOLD_1_066,CA_1,2011-02-21,0,7
4,HOUSEHOLD_1_066,CA_1,2011-02-28,0,7


In [41]:
first_positive_sale = (
    sales_dev_20.loc[sales_dev_20["units_sold"] > 0]
    .groupby("item_id")["date"]
    .min()
    .sort_values()
)

first_positive_sale

item_id
HOUSEHOLD_1_521   2011-01-29
HOUSEHOLD_1_118   2011-01-29
HOUSEHOLD_1_191   2011-01-29
HOUSEHOLD_1_303   2011-01-29
HOUSEHOLD_1_535   2011-02-15
HOUSEHOLD_1_151   2011-06-06
HOUSEHOLD_1_399   2011-07-29
HOUSEHOLD_1_066   2012-05-06
HOUSEHOLD_1_418   2012-06-01
HOUSEHOLD_1_447   2012-06-02
HOUSEHOLD_1_459   2012-06-02
HOUSEHOLD_1_465   2012-06-02
HOUSEHOLD_1_351   2012-06-02
HOUSEHOLD_1_334   2012-06-10
HOUSEHOLD_1_517   2013-01-25
HOUSEHOLD_1_409   2013-02-22
HOUSEHOLD_1_304   2013-02-22
HOUSEHOLD_1_110   2013-02-23
HOUSEHOLD_1_514   2013-02-23
HOUSEHOLD_1_096   2013-02-24
Name: date, dtype: datetime64[us]

In [42]:
prices_sample = pd.read_csv(
    "data/raw/sell_prices.csv",
    nrows=5
)

prices_sample

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [43]:
prices_scope = pd.DataFrame()

for block in pd.read_csv(
    "data/raw/sell_prices.csv",
    chunksize=100000
):
    selected = block.loc[
        (block["store_id"] == "CA_1") &
        (block["item_id"].isin(selected_items))
    ].copy()

    prices_scope = pd.concat(
        [prices_scope, selected],
        ignore_index=True
    )

print("Righe dei prezzi selezionate:", len(prices_scope))
prices_scope.head()

Righe dei prezzi selezionate: 4468


,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOUSEHOLD_1_066,11215,3.07
1,CA_1,HOUSEHOLD_1_066,11216,3.07
2,CA_1,HOUSEHOLD_1_066,11217,3.07
3,CA_1,HOUSEHOLD_1_066,11218,3.07
4,CA_1,HOUSEHOLD_1_066,11219,3.07


In [44]:
price_duplicates = prices_scope.duplicated(
    subset=["store_id", "item_id", "wm_yr_wk"]
).sum()

print("Combinazioni duplicate:", price_duplicates)
print("Prezzi mancanti:", prices_scope["sell_price"].isna().sum())
print("Prezzi non positivi:", (prices_scope["sell_price"] <= 0).sum())

Combinazioni duplicate: 0
Prezzi mancanti: 0
Prezzi non positivi: 0


In [45]:
sales_dev_priced = sales_dev_20.merge(
    calendar[["d", "wm_yr_wk"]],
    on="d",
    how="left",
    validate="many_to_one"
)

sales_dev_priced.head()

,item_id,store_id,d,units_sold,date,wm_yr_wk
0,HOUSEHOLD_1_066,CA_1,d_1,0,2011-01-29,11101
1,HOUSEHOLD_1_096,CA_1,d_1,0,2011-01-29,11101
2,HOUSEHOLD_1_110,CA_1,d_1,0,2011-01-29,11101
3,HOUSEHOLD_1_118,CA_1,d_1,3,2011-01-29,11101
4,HOUSEHOLD_1_151,CA_1,d_1,0,2011-01-29,11101


In [46]:
sales_dev_priced = sales_dev_priced.merge(
    prices_scope,
    on=["item_id", "store_id", "wm_yr_wk"],
    how="left",
    validate="many_to_one"
)

sales_dev_priced.head()

,item_id,store_id,d,units_sold,date,wm_yr_wk,sell_price
0,HOUSEHOLD_1_066,CA_1,d_1,0,2011-01-29,11101,NaN
1,HOUSEHOLD_1_096,CA_1,d_1,0,2011-01-29,11101,NaN
2,HOUSEHOLD_1_110,CA_1,d_1,0,2011-01-29,11101,NaN
3,HOUSEHOLD_1_118,CA_1,d_1,3,2011-01-29,11101,0.97
4,HOUSEHOLD_1_151,CA_1,d_1,0,2011-01-29,11101,NaN


In [47]:
missing_price = sales_dev_priced["sell_price"].isna()

print("Righe senza prezzo:", missing_price.sum())

print(
    "Righe con vendite positive senza prezzo:",
    (missing_price & (sales_dev_priced["units_sold"] > 0)).sum()
)

Righe senza prezzo: 8204
Righe con vendite positive senza prezzo: 0


In [ ]:
first_price_date = (
    sales_dev_priced.loc[
        sales_dev_priced["sell_price"].notna()
    ]
    .groupby("item_id")["date"].min()
)

first_price_per_row = sales_dev_priced["item_id"].map(first_price_date)

print(
    "Righe senza prezzo dalla prima data con prezzo in poi:",
    (
        missing_price &
        (sales_dev_priced["date"] >= first_price_per_row)
    ).sum()
)

Righe senza prezzo dalla prima data con prezzo in poi: 0


In [49]:
first_price_per_week = sales_weekly["item_id"].map(first_price_date)

sales_weekly_model = sales_weekly.loc[
    sales_weekly["week_start"] >= first_price_per_week
].copy()

print("Righe settimanali mantenute:", len(sales_weekly_model))
sales_weekly_model.head()

Righe settimanali mantenute: 4288


,item_id,store_id,week_start,weekly_units,days_present
66,HOUSEHOLD_1_066,CA_1,2012-05-07,27,7
67,HOUSEHOLD_1_066,CA_1,2012-05-14,36,7
68,HOUSEHOLD_1_066,CA_1,2012-05-21,27,7
69,HOUSEHOLD_1_066,CA_1,2012-05-28,12,7
70,HOUSEHOLD_1_066,CA_1,2012-06-04,17,7


In [50]:
validation_start = pd.Timestamp("2016-01-04")
validation_end = pd.Timestamp("2016-02-01")

train_first = sales_weekly_model.loc[
    sales_weekly_model["week_start"] < validation_start
].copy()

validation_first = sales_weekly_model.loc[
    (sales_weekly_model["week_start"] >= validation_start) &
    (sales_weekly_model["week_start"] < validation_end)
].copy()

print("Righe di validazione:", len(validation_first))

Righe di validazione: 80


In [52]:
validation_first

,item_id,store_id,week_start,weekly_units,days_present
257,HOUSEHOLD_1_066,CA_1,2016-01-04,57,7
258,HOUSEHOLD_1_066,CA_1,2016-01-11,45,7
259,HOUSEHOLD_1_066,CA_1,2016-01-18,48,7
260,HOUSEHOLD_1_066,CA_1,2016-01-25,42,7
530,HOUSEHOLD_1_096,CA_1,2016-01-04,77,7
...,...,...,...,...,...
5174,HOUSEHOLD_1_521,CA_1,2016-01-25,50,7
5444,HOUSEHOLD_1_535,CA_1,2016-01-04,49,7
5445,HOUSEHOLD_1_535,CA_1,2016-01-11,52,7
5446,HOUSEHOLD_1_535,CA_1,2016-01-18,35,7


In [53]:
last_week_sales = (
    train_first.sort_values("week_start")
    .groupby("item_id")
    .tail(1)
    .set_index("item_id")["weekly_units"]
)

validation_results = validation_first.copy()

validation_results["forecast_naive"] = (
    validation_results["item_id"].map(last_week_sales)
)

validation_results.head()

,item_id,store_id,week_start,weekly_units,days_present,forecast_naive
257,HOUSEHOLD_1_066,CA_1,2016-01-04,57,7,43
258,HOUSEHOLD_1_066,CA_1,2016-01-11,45,7,43
259,HOUSEHOLD_1_066,CA_1,2016-01-18,48,7,43
260,HOUSEHOLD_1_066,CA_1,2016-01-25,42,7,43
530,HOUSEHOLD_1_096,CA_1,2016-01-04,77,7,71


In [54]:
validation_results["error_naive"] = (
    validation_results["forecast_naive"] -
    validation_results["weekly_units"]
)

validation_results["absolute_error_naive"] = (
    validation_results["error_naive"].abs()
)

validation_results[
    [
        "item_id",
        "week_start",
        "weekly_units",
        "forecast_naive",
        "error_naive",
        "absolute_error_naive"
    ]
].head()

,item_id,week_start,weekly_units,forecast_naive,error_naive,absolute_error_naive
257,HOUSEHOLD_1_066,2016-01-04,57,43,-14,14
258,HOUSEHOLD_1_066,2016-01-11,45,43,-2,2
259,HOUSEHOLD_1_066,2016-01-18,48,43,-5,5
260,HOUSEHOLD_1_066,2016-01-25,42,43,1,1
530,HOUSEHOLD_1_096,2016-01-04,77,71,-6,6


In [55]:
mae_naive = validation_results["absolute_error_naive"].mean()

mean_error_naive = validation_results["error_naive"].mean()

print("MAE naïve:", round(mae_naive, 2))
print("Errore medio naïve:", round(mean_error_naive, 2))

MAE naïve: 14.98
Errore medio naïve: -1.52


In [56]:
mae_by_product = (
    validation_results.groupby("item_id")["absolute_error_naive"]
    .mean()
    .sort_values(ascending=False)
)

mae_by_product

item_id
HOUSEHOLD_1_303    36.75
HOUSEHOLD_1_191    35.25
HOUSEHOLD_1_521    30.25
HOUSEHOLD_1_118    23.75
HOUSEHOLD_1_459    21.75
HOUSEHOLD_1_409    18.50
HOUSEHOLD_1_351    16.25
HOUSEHOLD_1_418    13.75
HOUSEHOLD_1_110    12.25
HOUSEHOLD_1_465    11.00
HOUSEHOLD_1_304    10.75
HOUSEHOLD_1_447    10.75
HOUSEHOLD_1_399    10.00
HOUSEHOLD_1_334    10.00
HOUSEHOLD_1_517     8.50
HOUSEHOLD_1_514     6.50
HOUSEHOLD_1_535     6.25
HOUSEHOLD_1_096     6.00
HOUSEHOLD_1_151     5.75
HOUSEHOLD_1_066     5.50
Name: absolute_error_naive, dtype: float64

In [58]:
recent_four_weeks = (
    train_first.sort_values("week_start")
    .groupby("item_id")
    .tail(4)
)

mean_four_weeks = (
    recent_four_weeks.groupby("item_id")["weekly_units"]
    .mean()
)

validation_results["forecast_ma4"] = (
    validation_results["item_id"].map(mean_four_weeks)
)

validation_results[
    ["item_id", "week_start", "weekly_units",
     "forecast_naive", "forecast_ma4"]
].head(8)

,item_id,week_start,weekly_units,forecast_naive,forecast_ma4
257,HOUSEHOLD_1_066,2016-01-04,57,43,45.75
258,HOUSEHOLD_1_066,2016-01-11,45,43,45.75
259,HOUSEHOLD_1_066,2016-01-18,48,43,45.75
260,HOUSEHOLD_1_066,2016-01-25,42,43,45.75
530,HOUSEHOLD_1_096,2016-01-04,77,71,61.50
531,HOUSEHOLD_1_096,2016-01-11,70,71,61.50
532,HOUSEHOLD_1_096,2016-01-18,69,71,61.50
533,HOUSEHOLD_1_096,2016-01-25,56,71,61.50


In [59]:
validation_results["error_ma4"] = (
    validation_results["forecast_ma4"]
    - validation_results["weekly_units"]
)

validation_results["absolute_error_ma4"] = (
    validation_results["error_ma4"].abs()
)

mae_ma4 = validation_results["absolute_error_ma4"].mean()
mean_error_ma4 = validation_results["error_ma4"].mean()

print("MAE naïve:", round(mae_naive, 2))
print("MAE MA4:", round(mae_ma4, 2))
print("Errore medio MA4:", round(mean_error_ma4, 2))

MAE naïve: 14.98
MAE MA4: 11.99
Errore medio MA4: -7.04


In [60]:
comparison_by_product = (
    validation_results.groupby("item_id")[
        ["absolute_error_naive", "absolute_error_ma4"]
    ]
    .mean()
)

comparison_by_product["mae_reduction"] = (
    comparison_by_product["absolute_error_naive"]
    - comparison_by_product["absolute_error_ma4"]
)

comparison_by_product.sort_values("mae_reduction", ascending=False)

,absolute_error_naive,absolute_error_ma4,mae_reduction
item_id,,,
HOUSEHOLD_1_303,36.75,14.625,22.125
HOUSEHOLD_1_118,23.75,7.500,16.250
HOUSEHOLD_1_191,35.25,19.500,15.750
HOUSEHOLD_1_418,13.75,8.250,5.500
HOUSEHOLD_1_304,10.75,5.250,5.500
HOUSEHOLD_1_351,16.25,11.250,5.000
HOUSEHOLD_1_459,21.75,17.500,4.250
HOUSEHOLD_1_521,30.25,26.750,3.500
HOUSEHOLD_1_465,11.00,8.125,2.875


In [61]:
validation_starts = pd.date_range(
    start="2016-01-04",
    periods=4,
    freq="28D"
)

validation_starts

DatetimeIndex(['2016-01-04', '2016-02-01', '2016-02-29', '2016-03-28'],
              dtype='datetime64[us]', freq='28D')

In [62]:
for period_start in validation_starts:
    period_end = period_start + pd.Timedelta(days=28)

    train_period = sales_weekly_model.loc[
        sales_weekly_model["week_start"] < period_start
    ].copy()

    validation_period = sales_weekly_model.loc[
        (sales_weekly_model["week_start"] >= period_start)
        & (sales_weekly_model["week_start"] < period_end)
    ].copy()

    print(
        "Inizio:", period_start.date(),
        "| Fine esclusa:", period_end.date(),
        "| Righe validazione:", len(validation_period)
    )

Inizio: 2016-01-04 | Fine esclusa: 2016-02-01 | Righe validazione: 80
Inizio: 2016-02-01 | Fine esclusa: 2016-02-29 | Righe validazione: 80
Inizio: 2016-02-29 | Fine esclusa: 2016-03-28 | Righe validazione: 80
Inizio: 2016-03-28 | Fine esclusa: 2016-04-25 | Righe validazione: 80


In [63]:
backtest_blocks = []

for period_start in validation_starts:
    period_end = period_start + pd.Timedelta(days=28)

    train_period = sales_weekly_model.loc[
        sales_weekly_model["week_start"] < period_start
    ].copy()

    validation_period = sales_weekly_model.loc[
        (sales_weekly_model["week_start"] >= period_start)
        & (sales_weekly_model["week_start"] < period_end)
    ].copy()

    last_week = (
        train_period.sort_values("week_start")
        .groupby("item_id")
        .tail(1)
        .set_index("item_id")["weekly_units"]
    )

    last_four = (
        train_period.sort_values("week_start")
        .groupby("item_id")
        .tail(4)
    )

    mean_four = last_four.groupby("item_id")["weekly_units"].mean()

    validation_period["forecast_naive"] = (
        validation_period["item_id"].map(last_week)
    )

    validation_period["forecast_ma4"] = (
        validation_period["item_id"].map(mean_four)
    )

    validation_period["validation_start"] = period_start

    backtest_blocks.append(validation_period)

backtest_results = pd.concat(backtest_blocks, ignore_index=True)

print("Righe totali:", len(backtest_results))

Righe totali: 320


In [64]:
backtest_results["error_naive"] = (
    backtest_results["forecast_naive"]
    - backtest_results["weekly_units"]
)

backtest_results["error_ma4"] = (
    backtest_results["forecast_ma4"]
    - backtest_results["weekly_units"]
)

backtest_results["absolute_error_naive"] = (
    backtest_results["error_naive"].abs()
)

backtest_results["absolute_error_ma4"] = (
    backtest_results["error_ma4"].abs()
)

mae_by_period = (
    backtest_results.groupby("validation_start")[
        ["absolute_error_naive", "absolute_error_ma4"]
    ]
    .mean()
)

mae_by_period.round(2)

,absolute_error_naive,absolute_error_ma4
validation_start,,
2016-01-04,14.98,11.99
2016-02-01,12.32,10.74
2016-02-29,11.92,11.68
2016-03-28,17.12,14.99


In [65]:
mean_error_by_period = (
    backtest_results.groupby("validation_start")[
        ["error_naive", "error_ma4"]
    ]
    .mean()
)

mean_error_by_period.round(2)

,error_naive,error_ma4
validation_start,,
2016-01-04,-1.52,-7.04
2016-02-01,-3.85,-2.02
2016-02-29,0.10,0.05
2016-03-28,-6.48,-1.38


In [66]:
mae_naive_backtest = backtest_results["absolute_error_naive"].mean()
mae_ma4_backtest = backtest_results["absolute_error_ma4"].mean()

mae_improvement_pct = (
    (mae_naive_backtest - mae_ma4_backtest)
    / mae_naive_backtest
    * 100
)

print("MAE naïve complessivo:", round(mae_naive_backtest, 2))
print("MAE MA4 complessivo:", round(mae_ma4_backtest, 2))
print("Riduzione MAE (%):", round(mae_improvement_pct, 2))

MAE naïve complessivo: 14.09
MAE MA4 complessivo: 12.35
Riduzione MAE (%): 12.32


In [67]:
backtest_by_product = (
    backtest_results.groupby("item_id")[
        ["absolute_error_naive", "absolute_error_ma4"]
    ]
    .mean()
)

backtest_by_product["mae_reduction"] = (
    backtest_by_product["absolute_error_naive"]
    - backtest_by_product["absolute_error_ma4"]
)

backtest_by_product.sort_values(
    "mae_reduction", ascending=False
).round(2)

,absolute_error_naive,absolute_error_ma4,mae_reduction
item_id,,,
HOUSEHOLD_1_303,22.19,12.66,9.53
HOUSEHOLD_1_118,15.81,8.72,7.09
HOUSEHOLD_1_465,22.38,17.12,5.25
HOUSEHOLD_1_151,17.81,13.00,4.81
HOUSEHOLD_1_418,19.88,15.44,4.44
HOUSEHOLD_1_399,11.38,8.09,3.28
HOUSEHOLD_1_334,18.56,16.22,2.34
HOUSEHOLD_1_066,7.25,5.56,1.69
HOUSEHOLD_1_447,15.69,14.12,1.56


In [68]:
sales_test_20 = sales_test.loc[
    sales_test["item_id"].isin(selected_items)
].copy()

print("Righe:", len(sales_test_20))
print("Prima data:", sales_test_20["date"].min())
print("Ultima data:", sales_test_20["date"].max())

Righe: 560
Prima data: 2016-04-25 00:00:00
Ultima data: 2016-05-22 00:00:00


In [70]:
sales_test_20.to_sql(
    "sales_test_daily",
    conn,
    index=False,
    if_exists="replace"
)

query_test_weekly = """
SELECT
    item_id,
    store_id,
    DATE(date, '-6 days', 'weekday 1') AS week_start,
    SUM(units_sold) AS weekly_units,
    COUNT(DISTINCT DATE(date)) AS days_present
FROM sales_test_daily
GROUP BY item_id, store_id, week_start
HAVING COUNT(DISTINCT DATE(date)) = 7
ORDER BY item_id, store_id, week_start
"""

sales_test_weekly = pd.read_sql_query(
    query_test_weekly,
    conn,
    parse_dates=["week_start"]
)

print("Righe settimanali:", len(sales_test_weekly))
sales_test_weekly.head()

Righe settimanali: 80


,item_id,store_id,week_start,weekly_units,days_present
0,HOUSEHOLD_1_066,CA_1,2016-04-25,48,7
1,HOUSEHOLD_1_066,CA_1,2016-05-02,52,7
2,HOUSEHOLD_1_066,CA_1,2016-05-09,36,7
3,HOUSEHOLD_1_066,CA_1,2016-05-16,51,7
4,HOUSEHOLD_1_096,CA_1,2016-04-25,59,7


In [71]:
train_final = sales_weekly_model.loc[
    sales_weekly_model["week_start"] < test_start
].copy()

last_week_final = (
    train_final.sort_values("week_start")
    .groupby("item_id")
    .tail(1)
    .set_index("item_id")["weekly_units"]
)

last_four_final = (
    train_final.sort_values("week_start")
    .groupby("item_id")
    .tail(4)
)

mean_four_final = (
    last_four_final.groupby("item_id")["weekly_units"].mean()
)

test_results = sales_test_weekly.copy()

test_results["forecast_naive"] = (
    test_results["item_id"].map(last_week_final)
)

test_results["forecast_ma4"] = (
    test_results["item_id"].map(mean_four_final)
)

test_results[
    ["item_id", "week_start", "weekly_units",
     "forecast_naive", "forecast_ma4"]
].head()

,item_id,week_start,weekly_units,forecast_naive,forecast_ma4
0,HOUSEHOLD_1_066,2016-04-25,48,39,46.75
1,HOUSEHOLD_1_066,2016-05-02,52,39,46.75
2,HOUSEHOLD_1_066,2016-05-09,36,39,46.75
3,HOUSEHOLD_1_066,2016-05-16,51,39,46.75
4,HOUSEHOLD_1_096,2016-04-25,59,48,83.50


In [72]:
test_results["error_naive"] = (
    test_results["forecast_naive"] - test_results["weekly_units"]
)

test_results["error_ma4"] = (
    test_results["forecast_ma4"] - test_results["weekly_units"]
)

mae_naive_test = test_results["error_naive"].abs().mean()
mae_ma4_test = test_results["error_ma4"].abs().mean()

mean_error_ma4_test = test_results["error_ma4"].mean()

improvement_test_pct = (
    (mae_naive_test - mae_ma4_test) / mae_naive_test * 100
)

print("MAE naïve test:", round(mae_naive_test, 2))
print("MAE MA4 test:", round(mae_ma4_test, 2))
print("Errore medio MA4 test:", round(mean_error_ma4_test, 2))
print("Riduzione MAE sul test (%):", round(improvement_test_pct, 2))

MAE naïve test: 14.69
MAE MA4 test: 19.86
Errore medio MA4 test: 10.46
Riduzione MAE sul test (%): -35.23


In [73]:
test_by_product = (
    test_results.groupby("item_id")[
        ["error_naive", "error_ma4"]
    ]
    .agg(lambda errors: errors.abs().mean())
)

test_by_product["mae_increase"] = (
    test_by_product["error_ma4"]
    - test_by_product["error_naive"]
)

test_by_product.sort_values(
    "mae_increase", ascending=False
).round(2)

,error_naive,error_ma4,mae_increase
item_id,,,
HOUSEHOLD_1_334,39.00,75.50,36.50
HOUSEHOLD_1_096,7.50,28.00,20.50
HOUSEHOLD_1_447,10.75,28.75,18.00
HOUSEHOLD_1_110,8.00,19.75,11.75
HOUSEHOLD_1_351,7.50,16.75,9.25
HOUSEHOLD_1_303,10.50,18.00,7.50
HOUSEHOLD_1_514,6.50,13.50,7.00
HOUSEHOLD_1_304,3.50,7.50,4.00
HOUSEHOLD_1_399,6.75,10.38,3.62


In [74]:
history_334 = train_final.loc[
    (train_final["item_id"] == "HOUSEHOLD_1_334")
    & (train_final["week_start"] >= pd.Timestamp("2016-03-28")),
    ["week_start", "weekly_units"]
]

test_334 = test_results.loc[
    test_results["item_id"] == "HOUSEHOLD_1_334",
    ["week_start", "weekly_units", "forecast_naive", "forecast_ma4"]
]

display(history_334)
display(test_334)

,week_start,weekly_units
2453,2016-03-28,135
2454,2016-04-04,100
2455,2016-04-11,109
2456,2016-04-18,66


,week_start,weekly_units,forecast_naive,forecast_ma4
32,2016-04-25,29,66,102.5
33,2016-05-02,5,66,102.5
34,2016-05-09,31,66,102.5
35,2016-05-16,43,66,102.5


In [75]:
daily_test_334 = sales_test_20.loc[
    sales_test_20["item_id"] == "HOUSEHOLD_1_334",
    ["date", "units_sold"]
].sort_values("date")

daily_test_334

,date,units_sold
2003238,2016-04-25,10
2004285,2016-04-26,0
2005332,2016-04-27,4
2006379,2016-04-28,1
2007426,2016-04-29,6
2008473,2016-04-30,4
2009520,2016-05-01,4
2010567,2016-05-02,1
2011614,2016-05-03,0
2012661,2016-05-04,0


In [76]:
test_without_334 = test_results.loc[
    test_results["item_id"] != "HOUSEHOLD_1_334"
]

mae_without_334 = (
    test_without_334[["error_naive", "error_ma4"]]
    .abs()
    .mean()
)

mae_without_334.round(2)

error_naive    13.41
error_ma4      16.93
dtype: float64

In [77]:
report_columns = [
    "item_id", "store_id", "week_start", "weekly_units",
    "forecast_naive", "forecast_ma4", "error_naive", "error_ma4"
]

validation_report = backtest_results[report_columns].copy()
validation_report["phase"] = "Validation"

test_report = test_results[report_columns].copy()
test_report["phase"] = "Test"

forecast_report = pd.concat(
    [validation_report, test_report],
    ignore_index=True
)

forecast_report["phase"].value_counts()

phase
Validation    320
Test           80
Name: count, dtype: int64

In [79]:
validation_report.head()

,item_id,store_id,week_start,weekly_units,forecast_naive,forecast_ma4,error_naive,error_ma4,phase
0,HOUSEHOLD_1_066,CA_1,2016-01-04,57,43,45.75,-14,-11.25,Validation
1,HOUSEHOLD_1_066,CA_1,2016-01-11,45,43,45.75,-2,0.75,Validation
2,HOUSEHOLD_1_066,CA_1,2016-01-18,48,43,45.75,-5,-2.25,Validation
3,HOUSEHOLD_1_066,CA_1,2016-01-25,42,43,45.75,1,3.75,Validation
4,HOUSEHOLD_1_096,CA_1,2016-01-04,77,71,61.50,-6,-15.50,Validation


In [80]:
from pathlib import Path

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_file = output_dir / "forecast_report.csv"

forecast_report.to_csv(
    output_file,
    index=False,
    date_format="%Y-%m-%d"
)

print("File salvato in:", output_file.resolve())

File salvato in: /Users/federicocattin/Desktop/Demand planning/data/processed/forecast_report.csv


In [81]:
validation_windows = backtest_results.copy()
validation_windows["week_start"] = pd.to_datetime(validation_windows["week_start"])

first_week = validation_windows["week_start"].min()
days_from_start = (validation_windows["week_start"] - first_week).dt.days
validation_windows["window"] = (days_from_start // 28) + 1

validation_windows["abs_error_naive"] = validation_windows["error_naive"].abs()
validation_windows["abs_error_ma4"] = validation_windows["error_ma4"].abs()

mae_by_window = validation_windows.groupby("window").agg(
    first_week=("week_start", "min"),
    last_week=("week_start", "max"),
    n_rows=("week_start", "size"),
    mae_naive=("abs_error_naive", "mean"),
    mae_ma4=("abs_error_ma4", "mean"),
)

mae_by_window["delta_mae"] = mae_by_window["mae_ma4"] - mae_by_window["mae_naive"]

display(mae_by_window.round(2))

/var/folders/xw/_5wclt7d0cx3d45sgswnxfjc0000gn/T/ipykernel_58458/3559040959.py:21: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(mae_by_window.round(2))


,first_week,last_week,n_rows,mae_naive,mae_ma4,delta_mae
window,,,,,,
1,2016-01-04,2016-01-25,80,14.98,11.99,-2.98
2,2016-02-01,2016-02-22,80,12.32,10.74,-1.59
3,2016-02-29,2016-03-21,80,11.92,11.68,-0.24
4,2016-03-28,2016-04-18,80,17.12,14.99,-2.13


In [82]:
bias_by_phase = forecast_report.groupby("phase").agg(
    bias_naive=("error_naive", "mean"),
    bias_ma4=("error_ma4", "mean"),
)

display(bias_by_phase.round(2))

,bias_naive,bias_ma4
phase,,
Test,1.74,10.46
Validation,-2.94,-2.60


In [83]:
def forecast_ses(sales, alpha):
    level = sales[0]

    for actual in sales[1:]:
        level = alpha * actual + (1 - alpha) * level

    return level


forecast_ses([100, 80, 60], alpha=0.5)

75.0

In [84]:
%whos DataFrame

Variable                Type         Data/Info
----------------------------------------------
backtest_by_product     DataFrame    Shape: (20, 3)
backtest_results        DataFrame    Shape: (320, 12)
bias_by_phase           DataFrame    Shape: (2, 2)
block                   DataFrame    Shape: (41121, 4)
calendar                DataFrame    Shape: (1969, 14)
comparison_by_product   DataFrame    Shape: (20, 3)
daily_sales             DataFrame    Shape: (1047, 1941)
daily_test_334          DataFrame    Shape: (28, 2)
forecast_report         DataFrame    Shape: (400, 9)
history_334             DataFrame    Shape: (4, 2)
last_four               DataFrame    Shape: (80, 5)
last_four_final         DataFrame    Shape: (80, 5)
mae_by_period           DataFrame    Shape: (4, 2)
mae_by_window           DataFrame    Shape: (4, 6)
mean_error_by_period    DataFrame    Shape: (4, 2)
prices_sample           DataFrame    Shape: (5, 4)
prices_scope            DataFrame    Shape: (4468, 4)
products    

In [85]:
item = "HOUSEHOLD_1_334"
cutoff = pd.Timestamp("2016-01-04")

history = sales_dev_20.loc[
    sales_dev_20["item_id"] == item, ["date", "units_sold"]
].copy()

history["date"] = pd.to_datetime(history["date"])
history = history.loc[history["date"] < cutoff]
history["week_start"] = history["date"] - pd.to_timedelta(
    history["date"].dt.weekday, unit="D"
)

weekly_history = history.groupby("week_start", as_index=False).agg(
    weekly_units=("units_sold", "sum"),
    n_days=("date", "nunique"),
)
weekly_history = weekly_history.loc[weekly_history["n_days"] == 7]
weekly_history = weekly_history.sort_values("week_start")

sales = weekly_history["weekly_units"].tolist()
prediction = forecast_ses(sales, alpha=0.5)

first_window_334 = validation_windows.loc[
    (validation_windows["window"] == 1) & (validation_windows["item_id"] == item),
    ["week_start", "weekly_units", "forecast_naive", "forecast_ma4"],
].copy()

first_window_334["forecast_ses"] = prediction
display(first_window_334.sort_values("week_start").round(2))

/var/folders/xw/_5wclt7d0cx3d45sgswnxfjc0000gn/T/ipykernel_58458/3607536628.py:11: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  history["date"].dt.weekday, unit="D"
/var/folders/xw/_5wclt7d0cx3d45sgswnxfjc0000gn/T/ipykernel_58458/3607536628.py:30: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(first_window_334.sort_values("week_start").round(2))


,week_start,weekly_units,forecast_naive,forecast_ma4,forecast_ses
32,2016-01-04,107,101,81.75,85.74
33,2016-01-11,111,101,81.75,85.74
34,2016-01-18,105,101,81.75,85.74
35,2016-01-25,81,101,81.75,85.74


In [ ]:
# 1. Prepariamo le vendite settimanali dei 20 prodotti
daily = sales_dev_20[["item_id", "date", "units_sold"]].copy()
daily["date"] = pd.to_datetime(daily["date"])
daily["week_start"] = (
    daily["date"] - daily["date"].dt.weekday * pd.Timedelta(days=1)
)

weekly = daily.groupby(["item_id", "week_start"], as_index=False).agg(
    weekly_units=("units_sold", "sum"),
    n_days=("date", "nunique"),
)
weekly = weekly.loc[weekly["n_days"] == 7].sort_values("week_start")

# 2. Proviamo gli stessi valori di alpha per tutti i prodotti
alphas = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
results = []

for (window, item_id), target in validation_windows.groupby(["window", "item_id"]):
    cutoff = mae_by_window.loc[window, "first_week"]

    sales = weekly.loc[
        (weekly["item_id"] == item_id) & (weekly["week_start"] < cutoff),
        "weekly_units",
    ].tolist()

    for alpha in alphas:
        prediction = forecast_ses(sales, alpha)
        errors = prediction - target["weekly_units"]

        results.append({
            "alpha": alpha,
            "absolute_error_sum": errors.abs().sum(),
            "n_predictions": len(errors),
        })

# 3. Calcoliamo il MAE di validazione per ogni alpha
alpha_scores = pd.DataFrame(results).groupby("alpha").sum()
alpha_scores["mae_ses"] = (
    alpha_scores["absolute_error_sum"] / alpha_scores["n_predictions"]
)
alpha_scores = alpha_scores.sort_values("mae_ses")

display(alpha_scores[["n_predictions", "mae_ses"]].round(2))

In [89]:
# 1. Prepariamo le vendite settimanali dei 20 prodotti
daily = sales_dev_20[["item_id", "date", "units_sold"]].copy()
daily["date"] = pd.to_datetime(daily["date"])
daily["week_start"] = (
    daily["date"] - daily["date"].dt.weekday * pd.Timedelta(days=1)
)

weekly = daily.groupby(["item_id", "week_start"], as_index=False).agg(
    weekly_units=("units_sold", "sum"),
    n_days=("date", "nunique"),
)
weekly = weekly.loc[weekly["n_days"] == 7].sort_values("week_start")

# 2. Proviamo gli stessi valori di alpha per tutti i prodotti
alphas = [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
results = []

for (window, item_id), target in validation_windows.groupby(["window", "item_id"]):
    cutoff = mae_by_window.loc[window, "first_week"]

    sales = weekly.loc[
        (weekly["item_id"] == item_id) & (weekly["week_start"] < cutoff),
        "weekly_units",
    ].tolist()

    for alpha in alphas:
        prediction = forecast_ses(sales, alpha)
        errors = prediction - target["weekly_units"]

        results.append({
            "window": window,
            "alpha": alpha,
            "absolute_error_sum": errors.abs().sum(),
            "n_predictions": len(errors),
        })

# 3. Calcoliamo il MAE di validazione per ogni alpha
alpha_scores = pd.DataFrame(results).groupby("alpha").sum()
alpha_scores["mae_ses"] = (
    alpha_scores["absolute_error_sum"] / alpha_scores["n_predictions"]
)
alpha_scores = alpha_scores.sort_values("mae_ses")

display(alpha_scores[["n_predictions", "mae_ses"]].round(2))

/var/folders/xw/_5wclt7d0cx3d45sgswnxfjc0000gn/T/ipykernel_58458/3677907425.py:5: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  daily["date"] - daily["date"].dt.weekday * pd.Timedelta(days=1)


,n_predictions,mae_ses
alpha,,
0.3,320,11.94
0.1,320,12.15
0.5,320,12.35
0.7,320,12.91
0.9,320,13.63
1.0,320,14.09


In [90]:
best_alpha = alpha_scores["mae_ses"].idxmin()

ses_test = forecast_report.loc[forecast_report["phase"] == "Test"].copy()
test_start = pd.to_datetime(ses_test["week_start"]).min()

history_for_test = weekly.loc[weekly["week_start"] < test_start]

ses_by_item = history_for_test.groupby("item_id")["weekly_units"].agg(
    lambda sales: forecast_ses(sales.tolist(), best_alpha)
)

ses_test["forecast_ses"] = ses_test["item_id"].map(ses_by_item)
ses_test["error_ses"] = ses_test["forecast_ses"] - ses_test["weekly_units"]

test_errors = ses_test[["error_naive", "error_ma4", "error_ses"]]
test_errors.columns = ["Naive", "MA4", "SES"]

test_comparison = pd.DataFrame({
    "MAE": test_errors.abs().mean(),
    "Bias": test_errors.mean(),
})

print("Alpha scelto in validazione:", best_alpha)
print("Previsioni SES nel test:", ses_test["forecast_ses"].count())
display(test_comparison.round(2))

Alpha scelto in validazione: 0.3
Previsioni SES nel test: 80


,MAE,Bias
Naive,14.69,1.74
MA4,19.86,10.46
SES,18.04,8.42


In [91]:
ses_validation = validation_windows.copy()

for window, cutoff in mae_by_window["first_week"].items():
    history = weekly.loc[weekly["week_start"] < cutoff]

    predictions = history.groupby("item_id")["weekly_units"].agg(
        lambda sales: forecast_ses(sales.tolist(), best_alpha)
    )

    in_window = ses_validation["window"] == window
    ses_validation.loc[in_window, "forecast_ses"] = (
        ses_validation.loc[in_window, "item_id"].map(predictions)
    )

ses_validation["error_ses"] = (
    ses_validation["forecast_ses"] - ses_validation["weekly_units"]
)
ses_validation["abs_error_ses"] = ses_validation["error_ses"].abs()

window_comparison = mae_by_window[["mae_naive", "mae_ma4"]].copy()
window_comparison["mae_ses"] = (
    ses_validation.groupby("window")["abs_error_ses"].mean()
)

print("Previsioni SES in validazione:", ses_validation["forecast_ses"].count())
display(window_comparison.round(2))

Previsioni SES in validazione: 320


,mae_naive,mae_ma4,mae_ses
window,,,
1,14.98,11.99,11.77
2,12.32,10.74,10.58
3,11.92,11.68,10.79
4,17.12,14.99,14.62


In [92]:
scores_by_window = pd.DataFrame(results).groupby(
    ["window", "alpha"], as_index=False
).sum()

checks = []

for window in [3, 4]:
    past = scores_by_window.loc[scores_by_window["window"] < window]
    past_totals = past.groupby("alpha").sum()
    past_mae = past_totals["absolute_error_sum"] / past_totals["n_predictions"]
    chosen_alpha = past_mae.idxmin()

    current = scores_by_window.loc[
        (scores_by_window["window"] == window)
        & (scores_by_window["alpha"] == chosen_alpha)
    ].iloc[0]

    checks.append({
        "window": window,
        "chosen_alpha": chosen_alpha,
        "mae_ses": current["absolute_error_sum"] / current["n_predictions"],
        "mae_naive": mae_by_window.loc[window, "mae_naive"],
        "mae_ma4": mae_by_window.loc[window, "mae_ma4"],
    })

display(pd.DataFrame(checks).round(2))

,window,chosen_alpha,mae_ses,mae_naive,mae_ma4
0,3,0.1,11.71,11.92,11.68
1,4,0.3,14.62,17.12,14.99


In [93]:
error_cols = ["error_naive", "error_ma4", "error_ses"]

four_week_errors = ses_test.groupby("item_id")[error_cols].sum()

four_week_mae = four_week_errors.abs().mean()
four_week_mae.index = ["Naive", "MA4", "SES"]

display(four_week_mae.rename("MAE sul totale di 4 settimane").round(2))

Naive    45.15
MA4      65.75
SES      55.13
Name: MAE sul totale di 4 settimane, dtype: float64

In [94]:
item_errors = four_week_errors[["error_naive", "error_ses"]].abs()

item_errors["ses_minus_naive"] = (
    item_errors["error_ses"] - item_errors["error_naive"]
)

delta = item_errors["ses_minus_naive"]

print("Prodotti in cui Naive è più preciso:", (delta > 0).sum())
print("Prodotti in cui SES è più preciso:", (delta < 0).sum())
print("Pareggi:", (delta == 0).sum())

display(item_errors.sort_values("ses_minus_naive", ascending=False).round(2))

Prodotti in cui Naive è più preciso: 11
Prodotti in cui SES è più preciso: 9
Pareggi: 0


,error_naive,error_ses,ses_minus_naive
item_id,,,
HOUSEHOLD_1_334,156,274.86,118.86
HOUSEHOLD_1_096,30,92.25,62.25
HOUSEHOLD_1_447,43,105.06,62.06
HOUSEHOLD_1_351,18,77.83,59.83
HOUSEHOLD_1_303,18,73.92,55.92
HOUSEHOLD_1_110,32,63.89,31.89
HOUSEHOLD_1_399,1,24.31,23.31
HOUSEHOLD_1_514,18,39.84,21.84
HOUSEHOLD_1_304,4,25.50,21.50


In [95]:
most_favorable_item = delta.idxmax()

gap_all = delta.mean()
gap_without_item = delta.drop(index=most_favorable_item).mean()

print("Prodotto che favorisce di più Naive:", most_favorable_item)
print(f"Vantaggio medio di Naive, tutti i prodotti: {gap_all:.2f}")
print(f"Vantaggio medio di Naive, senza quel prodotto: {gap_without_item:.2f}")

Prodotto che favorisce di più Naive: HOUSEHOLD_1_334
Vantaggio medio di Naive, tutti i prodotti: 9.98
Vantaggio medio di Naive, senza quel prodotto: 4.25


In [96]:
from pathlib import Path
import pandas as pd

report_columns = [
    "item_id", "store_id", "week_start", "weekly_units",
    "forecast_naive", "forecast_ma4", "error_naive", "error_ma4",
    "phase", "forecast_ses", "error_ses",
]

forecast_report_with_ses = pd.concat(
    [
        ses_validation.assign(phase="Validation")[report_columns],
        ses_test.assign(phase="Test")[report_columns],
    ],
    ignore_index=True,
)

assert len(forecast_report_with_ses) == 400, "Il report deve avere 400 righe."
assert forecast_report_with_ses[["forecast_ses", "error_ses"]].notna().all().all(), (
    "Ci sono valori SES mancanti."
)

output_path = Path("forecast_report_with_ses.csv").resolve()
forecast_report_with_ses.to_csv(output_path, index=False)

print(f"File creato: {output_path}")
print(forecast_report_with_ses["phase"].value_counts())

File creato: /Users/federicocattin/Desktop/Demand planning/forecast_report_with_ses.csv
phase
Validation    320
Test           80
Name: count, dtype: int64


Nelle quattro settimane di test, Naive ha ottenuto il minore errore assoluto medio sul totale venduto per prodotto: 45,15 unità, contro 55,13 per SES e 65,75 per MA4. Nel confronto diretto, Naive è stato più preciso su 11 prodotti e SES su 9. Escludendo il prodotto più favorevole a Naive, il vantaggio medio scende da 9,98 a 4,25 unità, ma resta positivo. Il risultato riguarda i 20 prodotti e il periodo esaminati.